In [1]:
!nvidia-smi

Tue Mar 17 16:07:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install accelerate
!pip install "transformers==4.43.0" "huggingface_hub==0.34.1" bitsandbytes


In [3]:
from huggingface_hub import list_repo_files
# finds possible 7b qwen 2.5 models
files = list_repo_files("Qwen/Qwen2.5-7B-Instruct-GGUF")
for f in files:
    if f.endswith(".gguf"):
        print(f)


qwen2.5-7b-instruct-fp16-00001-of-00004.gguf
qwen2.5-7b-instruct-fp16-00002-of-00004.gguf
qwen2.5-7b-instruct-fp16-00003-of-00004.gguf
qwen2.5-7b-instruct-fp16-00004-of-00004.gguf
qwen2.5-7b-instruct-q2_k.gguf
qwen2.5-7b-instruct-q3_k_m.gguf
qwen2.5-7b-instruct-q4_0-00001-of-00002.gguf
qwen2.5-7b-instruct-q4_0-00002-of-00002.gguf
qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf
qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf
qwen2.5-7b-instruct-q5_0-00001-of-00002.gguf
qwen2.5-7b-instruct-q5_0-00002-of-00002.gguf
qwen2.5-7b-instruct-q5_k_m-00001-of-00002.gguf
qwen2.5-7b-instruct-q5_k_m-00002-of-00002.gguf
qwen2.5-7b-instruct-q6_k-00001-of-00002.gguf
qwen2.5-7b-instruct-q6_k-00002-of-00002.gguf
qwen2.5-7b-instruct-q8_0-00001-of-00003.gguf
qwen2.5-7b-instruct-q8_0-00002-of-00003.gguf
qwen2.5-7b-instruct-q8_0-00003-of-00003.gguf


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from huggingface_hub import hf_hub_download
# downloads 4 bit quantized qwen 2.5 7b model
repo_id = "Qwen/Qwen2.5-7B-Instruct-GGUF"

files = [
    "qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf",
    "qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf",
]

paths = []
for f in files:
    paths.append(
        hf_hub_download(repo_id=repo_id, filename=f)
    )

paths


['/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct-GGUF/snapshots/bb5d59e06d9551d752d08b292a50eb208b07ab1f/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf',
 '/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct-GGUF/snapshots/bb5d59e06d9551d752d08b292a50eb208b07ab1f/qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf']

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

# Load model (auto places on GPU/CPU)
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)

# Tokenize
inputs = tokenizer("Hello world", return_tensors="pt")

# Move inputs to the correct device (first device the model uses)
device = next(model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

# Generate
outputs = model.generate(**inputs, max_new_tokens=50)

# Decode
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

ValueError: `.to` is not supported for `4-bit` or `8-bit` bitsandbytes models. Please use the model as it is, since the model has already been set to the correct devices and casted to the correct `dtype`.

In [ ]:
# input
inputs = tokenizer("Hello world", return_tensors="pt")

# output in tokens
outputs = model.generate(**inputs, max_new_tokens=50)

# Decode
print(tokenizer.decode(outputs[0]))
